# Marconi daily DEM movie
csherwood@usgs.gov, 2026-10-26, generated with Claude Opus 5.5 (survey dates and pixel instruments added 2026-10-27)

Regenerates a five-panel frame for each day from the GeoTIFFs written by `make_dems.ipynb`, then assembles the
frames into a Motion-JPEG .avi:
* a, b: Hs and water level at the nanopod (static), with a red bar at the DEM day, its three-day window shaded,
  and thick light gray lines at the beach-survey dates (`mm.SURVEYS`);
* c: DEM with labeled 1 m contours;
* d: change from the previous non-empty DEM (blue accretion, red erosion);
* e: uncertainty.

Panels c–e share the same scale and bounds (`MOVIE['xlim']`, `MOVIE['ylim']`; None = data extent) and show the
pixel-instrument transects from `pix_xyz_daily.csv` (written by `qc_waterlines.ipynb`) if that file exists.

Requires: numpy, pandas, matplotlib, opencv, rasterio.

In [ ]:
from pathlib import Path
import pandas as pd
import marconi_movie as mm

# --- config
PROJ = Path(r'F:\crs\proj\2026_wl_dems')
WAVES = PROJ / 'nanopod_waves' / 'sig1000_waves_ALL1.csv'
OUT_DIR = PROJ / 'scan'
DEM_DIR = OUT_DIR / 'dems'
PIX_CSV = OUT_DIR / 'pix_xyz_daily.csv'
FIG_DIR = PROJ / 'fig'
FRAME_DIR = FIG_DIR / 'movie_frames'
AVI = FIG_DIR / 'marconi_dems.avi'
FPS = 4
FIRST_DAY, LAST_DAY = None, None     # e.g. '2024-12-10', '2025-02-15'; None = all days in dem_days.csv
MOVIE = mm.MOVIE | {'xlim': (None, 420200.)}

FRAME_DIR.mkdir(parents=True, exist_ok=True)
waves = pd.read_csv(WAVES, parse_dates=['time'])
dt = pd.read_csv(OUT_DIR / 'dem_days.csv', parse_dates=['day'])
dt = dt[(dt.day >= pd.Timestamp(FIRST_DAY or dt.day.min())) & (dt.day <= pd.Timestamp(LAST_DAY or dt.day.max()))]
status = {r.day: f"{r.status}, {r.n_img_used:.0f} images, slope {r.slope_median:.3f}" if r.status == 'ok' else r.status
          for r in dt.itertuples()}
pix = pd.read_csv(PIX_CSV) if PIX_CSV.exists() else None
print(len(dt), 'days;', mm.data_bounds(DEM_DIR, dt.day, MOVIE), '; pixel instruments' if pix is not None else '')

In [ ]:
paths = mm.make_frames(DEM_DIR, FRAME_DIR, list(dt.day), waves, status, MOVIE, pix)
print(len(paths), 'frames')

In [ ]:
paths = sorted(FRAME_DIR.glob('frame_*.png'))
mm.write_avi(paths, AVI, FPS)
print(f'{len(paths)} frames -> {AVI}, {AVI.stat().st_size / 1e6:.1f} MB')